# Training Capture v2 Soundness Quickcheck

This notebook performs a focused validation of the new during-training capture methodology.

## What it checks

1. **Bundle/schema safety**
   - `run_id` separation (`__s2` style)
   - expected `settings.json` flags
   - presence/shape consistency of new `during_*` keys
2. **Methodological soundness (hypothesis-aligned)**
   - During **B phase** (`feature_probe==1`): comms balance proxy `m0/m1`
   - During **A2 phase**: compare probe 0 vs probe 1 comms balance
3. **Dynamic functional specialization proxy**
   - Sliding-window module hidden norm ratio from `hiddens_per_module`
4. **Quick integrity checks**
   - finite-value rates, alignment with trial masks, expected dimensions

Use this as a go/no-go check before full-scale runs.

In [3]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.analysis.during_training import (
    summarize_during_npz,
    phase_B_probe1_comms_ratio,
    phase_A2_by_probe,
)

print("Project root:", project_root)

Project root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular


In [ ]:
# ---- configure ----
# Option A: set RUN_ID directly to an existing folder under data/simulations
# Option B: leave RUN_ID empty and set CONDITION_NAME to auto-compute run_id from experiments.json
RUN_ID = ""
CONDITION_NAME = "two_module_rnn_50_task_routed_trainlog_s2"

PARTICIPANT = None  # e.g. "study1_same_sub1"; None -> auto-pick first sim_*.npz
PHASE_B = 1
PHASE_A2 = 2
ROLLING_WINDOW = 25

from a1b2.utils.run_config import build_run_id

sim_root = project_root / "data" / "simulations"

if not RUN_ID:
    cfg_path = project_root / "a1b2" / "models" / "experiments.json"
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    cond = next((c for c in cfg.get("conditions", []) if c.get("name") == CONDITION_NAME), None)
    assert cond is not None, f"Condition not found in experiments.json: {CONDITION_NAME}"
    RUN_ID = build_run_id(cond)
    print("Auto-computed RUN_ID from condition:", RUN_ID)

sim_folder = sim_root / RUN_ID
settings_path = sim_folder / "settings.json"

if not sim_folder.exists():
    nearby = sorted([p.name for p in sim_root.glob(f"*{CONDITION_NAME}*")])
    msg = f"Run folder not found: {sim_folder}"
    if nearby:
        msg += "\n\nNearby folders:\n- " + "\n- ".join(nearby[:20])
    msg += "\n\nIf this is a new condition, run simulations first, then rerun this cell."
    raise AssertionError(msg)

assert settings_path.exists(), f"settings.json not found in: {sim_folder}"

AssertionError: Run folder not found: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations/two_module_rnn_50_task_routed_trainlog_s2__s2

In [ ]:
# 1) settings + run-level safety checks
with open(settings_path, "r") as f:
    settings = json.load(f)

print("sim_folder:", sim_folder)
print("sim_schema_version:", settings.get("sim_schema_version"))
print("log_during_training:", settings.get("log_during_training"))
print("during_log_post_step:", settings.get("during_log_post_step", "<missing=>default True>"))
print("dataloader_num_workers:", settings.get("dataloader_num_workers"))

if "__s2" not in sim_folder.name:
    print("WARNING: run_id does not contain '__s2'. Confirm separation from legacy runs.")

npz_files = sorted(sim_folder.glob("sim_*.npz"))
assert npz_files, f"No sim_*.npz files in {sim_folder}"
if PARTICIPANT is None:
    npz_path = npz_files[0]
else:
    npz_path = sim_folder / f"sim_{PARTICIPANT}.npz"
    assert npz_path.exists(), f"NPZ not found: {npz_path}"

print("Using NPZ:", npz_path.name)

In [ ]:
# 2) load participant NPZ + schema integrity checks
with np.load(npz_path, allow_pickle=True) as z:
    data = {k: z[k] for k in z.files}

required_legacy = ["losses", "accuracy", "probes", "hiddens_per_module"]
required_during = [
    "during_core_per_module",
    "during_comms_per_module",
    "during_core_l2",
    "during_comms_l2",
]

missing_legacy = [k for k in required_legacy if k not in data]
missing_during = [k for k in required_during if k not in data]

print("Missing legacy keys:", missing_legacy)
print("Missing during keys:", missing_during)
assert not missing_legacy, f"Missing required legacy keys: {missing_legacy}"
assert not missing_during, f"Missing required during keys: {missing_during}"

for k in required_legacy + required_during:
    print(f"{k:>26}: {data[k].shape}")

# consistency checks
assert data["hiddens_per_module"].shape[:2] == data["during_core_per_module"].shape[:2]
assert data["during_core_per_module"].shape == data["during_comms_per_module"].shape
assert data["during_core_l2"].shape == data["during_comms_l2"].shape
assert data["during_core_l2"].shape[:2] == data["during_core_per_module"].shape[:2]

summary = summarize_during_npz(data)
pd.Series(summary)

In [ ]:
# helper utilities

def valid_phase_mask(losses_2d, phase):
    return np.isfinite(losses_2d[phase])

def rolling_mean_1d(x, w=25):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return x
    w = max(1, min(int(w), x.size))
    s = pd.Series(x)
    return s.rolling(window=w, min_periods=max(1, w // 4)).mean().to_numpy()

def module_norm_ratio_from_hpm(hpm_phase, eps=1e-8):
    # hpm_phase: (n_trials, n_modules, hidden_size)
    n = np.linalg.norm(hpm_phase, axis=-1)  # (n_trials, n_modules)
    if n.shape[1] < 2:
        return np.full(n.shape[0], np.nan, dtype=np.float32)
    return (n[:, 0] / (n[:, 1] + eps)).astype(np.float32)

In [ ]:
# 3) hypothesis-aligned comms checks (B and A2)
losses = data["losses"]
probes = data["probes"]
during_comms_l2 = data["during_comms_l2"]

b = phase_B_probe1_comms_ratio(losses, probes, during_comms_l2, phase_b=PHASE_B)
a2 = phase_A2_by_probe(losses, probes, during_comms_l2, phase_a2=PHASE_A2)

print("B probe=1 n:", b["n_trials"])
print("A2 probe0 n:", a2["probe_0_n"], "| probe1 n:", a2["probe_1_n"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(b["comms_m0_over_m1"], bins=30)
axes[0].axvline(np.nanmedian(b["comms_m0_over_m1"]), linestyle="--")
axes[0].set_title("B phase (probe=1): comms m0/m1")
axes[0].set_xlabel("ratio")
axes[0].set_ylabel("count")

box_data = [a2["probe_0_comms_m0_over_m1"], a2["probe_1_comms_m0_over_m1"]]
axes[1].boxplot(box_data, labels=["A2 probe0", "A2 probe1"], showfliers=False)
axes[1].set_title("A2: comms m0/m1 by probe")
axes[1].set_ylabel("ratio")

plt.tight_layout()
plt.show()

print("B median ratio:", float(np.nanmedian(b["comms_m0_over_m1"])))
print("A2 probe0 median:", float(np.nanmedian(a2["probe_0_comms_m0_over_m1"])))
print("A2 probe1 median:", float(np.nanmedian(a2["probe_1_comms_m0_over_m1"])))

In [ ]:
# 4) dynamic functional-specialization proxy over training
hpm = data["hiddens_per_module"]  # (phase, trials, modules, hidden)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)
phase_names = {0: "A1", 1: "B", 2: "A2"}

for phase in [0, 1, 2]:
    m = valid_phase_mask(losses, phase)
    if m.sum() == 0:
        axes[phase].set_title(f"{phase_names[phase]}: no valid trials")
        continue

    ratio_h = module_norm_ratio_from_hpm(hpm[phase, m])
    ratio_c = (during_comms_l2[phase, m, 0] / (during_comms_l2[phase, m, 1] + 1e-8)).astype(np.float32)
    p = probes[phase, m]

    x = np.arange(m.sum())
    axes[phase].plot(x, rolling_mean_1d(ratio_h, ROLLING_WINDOW), label="hidden m0/m1 (rolling)")
    axes[phase].plot(x, rolling_mean_1d(ratio_c, ROLLING_WINDOW), label="comms m0/m1 (rolling)")

    # phase-specific probe markers
    probe0 = np.where(p == 0)[0]
    probe1 = np.where(p == 1)[0]
    if probe0.size:
        axes[phase].scatter(probe0, np.full_like(probe0, np.nanmin(ratio_h)), s=5, alpha=0.25, label="probe0")
    if probe1.size:
        axes[phase].scatter(probe1, np.full_like(probe1, np.nanmax(ratio_h)), s=5, alpha=0.25, label="probe1")

    axes[phase].set_title(f"{phase_names[phase]}: dynamic ratio traces")
    axes[phase].set_ylabel("ratio")
    axes[phase].legend(loc="upper right")

axes[-1].set_xlabel("valid trial index within phase")
plt.tight_layout()
plt.show()

In [ ]:
# 5) machine-readable mini report (for quick comparison across runs)
report = {
    "run_id": RUN_ID,
    "npz_file": npz_path.name,
    "sim_schema_version": settings.get("sim_schema_version"),
    "log_during_training": settings.get("log_during_training"),
    "during_log_post_step": settings.get("during_log_post_step", True),
    "has_during_comms_ratio_key": "during_comms_m0_over_m1_l2" in data,
    "b_probe1_n": int(b["n_trials"]),
    "b_probe1_median_comms_m0_m1": float(np.nanmedian(b["comms_m0_over_m1"])),
    "a2_probe0_n": int(a2["probe_0_n"]),
    "a2_probe1_n": int(a2["probe_1_n"]),
    "a2_probe0_median_comms_m0_m1": float(np.nanmedian(a2["probe_0_comms_m0_over_m1"])),
    "a2_probe1_median_comms_m0_m1": float(np.nanmedian(a2["probe_1_comms_m0_over_m1"])),
}

report_path = sim_folder / f"soundness_quickcheck__{npz_path.stem}.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

pd.Series(report)
print("Saved:", report_path)